# New h5pyd / HSDS features

This notebook tours some of the newer h5pyd/HSDS capabilities that aren't
covered by the other example notebooks in this directory:

* array dtypes (bare, and as a compound field)
* compound field-selective read/write (write isolation)
* boolean mask selection
* point selection via the `.points` accessor
* server-side `Dataset.query()` with `update_value=`
* region reference `.query()`
* the `Table` class (`create_table`, `read_where`, `update_where`)
* `track_order` on groups


In [19]:
import numpy as np
import h5pyd as h5py

# change if needed to whatever folder path works for you
filepath = "/home/test_user1/test/new_features_example.h5"
f = h5py.File(filepath, "w")
f

<HDF5 file "new_features_example.h5" (mode r+)>

## Array dtypes

Datasets can use a bare array (subarray) dtype, or a compound dtype with an array-typed field.

In [20]:
# a bare array dtype: each element of the dataset is itself a 3-element int32 array
dt_vec3 = np.dtype(("i4", (3,)))
dset = f.create_dataset("bare_array", (4,), dtype=dt_vec3)
dset[0] = [1, 2, 3]
dset[1:3] = [[4, 5, 6], [7, 8, 9]]
dset[:]

array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9],
       [0, 0, 0]], dtype=int32)

In [21]:
# a compound dtype with an array-typed field
dt_particle = np.dtype([("id", "i4"), ("velocity", "f4", (3,))])
dset = f.create_dataset("compound_array_field", (3,), dtype=dt_particle)
arr = np.zeros((3,), dtype=dt_particle)
arr["id"] = [1, 2, 3]
arr["velocity"] = [[1, 1, 1], [2, 2, 2], [3, 3, 3]]
dset[:] = arr
dset[:]

array([(1, [1., 1., 1.]), (2, [2., 2., 2.]), (3, [3., 3., 3.])],
      dtype=[('id', '<i4'), ('velocity', '<f4', (3,))])

## Compound field-selective read/write

Reading or writing a subset of a compound dataset's fields no longer disturbs
the fields that weren't selected.

In [22]:
dt_rec = np.dtype([("a", "i4"), ("b", "i4"), ("c", "i4")])
dset = f.create_dataset("field_isolation", (3,), dtype=dt_rec)
dset[:] = np.zeros((3,), dtype=dt_rec)

# write only fields "a" and "c" -- "b" should be left untouched
dset["a", "c"] = np.array([(1, 10), (2, 20), (3, 30)], dtype=[("a", "i4"), ("c", "i4")])
dset[:]

array([(1, 0, 10), (2, 0, 20), (3, 0, 30)],
      dtype=[('a', '<i4'), ('b', '<i4'), ('c', '<i4')])

In [23]:
# reading a field subset returns just those fields
dset["a", "c"][:]

array([(1, 10), (2, 20), (3, 30)], dtype=[('a', '<i4'), ('c', '<i4')])

## Boolean mask selection

Datasets can be indexed directly with a boolean array, same as NumPy.

In [24]:
dset = f.create_dataset("bool_mask", (10,), dtype="i4", data=np.arange(10))
mask = dset[:] % 2 == 0
dset[mask]

array([0, 2, 4, 6, 8], dtype=int32)

## Point selection via `.points`

`Dataset.points` reads or writes an arbitrary, unordered, repeatable list of
coordinates in a single round trip.  Note: this is not currently supported in h5py

In [25]:
dset = f.create_dataset("points", (2, 5), dtype="i4", data=np.arange(10).reshape(2, 5))
pts = [(0, 4), (1, 0), (0, 4), (1, 2)]  # note: repeated and out-of-order points
dset.points[pts]

array([4, 5, 4, 7], dtype=int32)

In [26]:
dset.points[pts] = [100, 200, 300, 400]
dset[:]

array([[  0,   1,   2,   3, 300],
       [200,   6, 400,   8,   9]], dtype=int32)

## Server-side query with `update_value`

`Dataset.query()` pushes a boolean expression to HSDS and returns the
matching indices. Passing `update_value=` additionally updates the matched
elements server-side, without pulling the whole dataset locally first.

In [27]:
dt_xy = np.dtype([("x", "i4"), ("y", "i4")])
dset = f.create_dataset("query_ds", (5,), dtype=dt_xy)
dset[:] = np.array([(i, i * 10) for i in range(5)], dtype=dt_xy)
dset[:]

dset.query("x > 2")

array([[3],
       [4]])

In [28]:
dset.query("x > 2", update_value={"y": -1})
dset[:]

array([(0,  0), (1, 10), (2, 20), (3, -1), (4, -1)],
      dtype=[('x', '<i4'), ('y', '<i4')])

## Region reference `.query()`

`dset.regionref.query()` builds a point-selection region reference from a
query, which can then be used to index the dataset (or stored as an
attribute for later use).

In [29]:
dset = f.create_dataset("region_src", (5,), dtype=dt_xy,
                        data=np.array([(i, i * 10) for i in range(5)], dtype=dt_xy))
regref = dset.regionref.query("x > 2")
dset[regref]

array([(3, 30), (4, 40)], dtype=[('x', '<i4'), ('y', '<i4')])

## The `Table` class

`Group.create_table()` returns a PyTables-style `Table` over a compound
dataset, supporting server-side `read_where()`/`update_where()` queries.

In [30]:
table = f.create_table("mytable", numrows=5, dtype=dt_xy)
table[:] = np.array([(i, i * 10) for i in range(5)], dtype=dt_xy)
list(table.read_where("x > 2"))

[np.void((3, 30), dtype=[('x', '<i4'), ('y', '<i4')]),
 np.void((4, 40), dtype=[('x', '<i4'), ('y', '<i4')])]

In [31]:
table.update_where("x > 2", {"y": -1})
table[:]

array([(0,  0), (1, 10), (2, 20), (3, -1), (4, -1)],
      dtype=[('x', '<i4'), ('y', '<i4')])

## `track_order`

Groups can be created with `track_order=True` so that member iteration
follows creation order rather than alphanumeric order.

In [32]:
grp = f.create_group("ordered_group", track_order=True)
for name in ["zebra", "apple", "mango"]:
    grp.create_group(name)
list(grp.keys())

['zebra', 'apple', 'mango']

## Opaque data

HDF5's opaque type stores raw, fixed-size, uninterpreted bytes -- useful for
arbitrary binary blobs HDF5 doesn't need to understand. It's represented in
NumPy as an unstructured `void` dtype (`"V<n>"`, no fields).

In [33]:
dt_blob = np.dtype("V16")
dset = f.create_dataset("blobs", (3,), dtype=dt_blob)
blobs = [b"\x00\x01\xffhello!!!", b"binary-blob-data", b"\x99\x88\x77zzzzzzzz"]
for i, b in enumerate(blobs):
    dset[i] = np.void(b.ljust(16, b"\x00")[:16])
dset.dtype

dtype('V16')

In [34]:
# read the raw bytes back
[bytes(v) for v in dset[:]]

[b'\x00\x01\xffhello!!!\x00\x00\x00\x00\x00',
 b'binary-blob-data',
 b'\x99\x88wzzzzzzzz\x00\x00\x00\x00\x00']

In [35]:
# opaque data works as an attribute too
f.attrs.create("opaque_attr", np.void(b"attr-blob-value!"), dtype="V16")
bytes(f.attrs["opaque_attr"])

b'attr-blob-value!'

> [!NOTE]
> h5py additionally offers `h5py.opaque_dtype()` -- a convenience helper
> that tags an arbitrary dtype (e.g. `datetime64`) so it's stored as opaque
> data, letting you round-trip types HDF5 has no native equivalent for.
> h5pyd currently exports the corresponding checker, `check_opaque_dtype()`,
> but not the `opaque_dtype()` constructor itself, and datetime64/timedelta64
> arrays aren't yet routed through the opaque encoding path -- so that
> specific convenience (as opposed to plain `"V<n>"` opaque data, above,
> which works fine) isn't available yet.

In [ ]:
f.close()